# WORK9 — 05C Soft Two-Part Expected-Demand Challenger V01

**Research-only targeted challenger.** It does not rebuild Dataset V012, Feature V013, or Feature Selection V04.

Formula:

$$\hat y = P(Y>0\mid X) \times E[Y\mid Y>0,X]$$

There is **no hard probability threshold**, no Pair-level ceil, no global bias scaling, no Frozen Test, no freeze, no production publish.

Use a **GPU runtime** in Colab, then Run all.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install lightgbm pyarrow pyyaml pytest

from pathlib import Path
import json, sys, subprocess, datetime
import pandas as pd

ROOT = Path('/content/drive/MyDrive/work9')
SRC = ROOT / '02_src/modeling/soft_two_part_challenger_runner_v01.py'
TEST = ROOT / '07_tests/test_soft_two_part_challenger_runner_v01.py'
CHAL_CONTRACT = ROOT / '01_config/soft_two_part_challenger_contract_v01.yaml'
MODEL_CONTRACT = ROOT / '01_config/model_contract_v02.yaml'
ROLL_CONTRACT = ROOT / '01_config/rolling_backtest_contract_v01.yaml'
DATA_PTR = ROOT / '01_config/current_dataset_run.json'
FEAT_PTR = ROOT / '01_config/current_feature_run.json'
SEL_PTR = ROOT / '01_config/current_feature_selection_run.json'
ROLL_PTR = ROOT / '01_config/current_rolling_backtest_run.json'
DIAG_PTR = ROOT / '01_config/current_underforecast_diagnosis_run.json'

for p in [SRC, TEST, CHAL_CONTRACT, MODEL_CONTRACT, ROLL_CONTRACT, DATA_PTR, FEAT_PTR, SEL_PTR, ROLL_PTR, DIAG_PTR]:
    assert p.exists(), f'Missing: {p}'
print('05C inputs found')

Mounted at /content/drive
05C inputs found


## 1. Lock accepted lineage
05C must use the same Dataset V012 → Feature V013 → Feature Selection V04 → Rolling V01 lineage already audited.

In [2]:
dataset = json.loads(DATA_PTR.read_text(encoding='utf-8'))
feature = json.loads(FEAT_PTR.read_text(encoding='utf-8'))
selection = json.loads(SEL_PTR.read_text(encoding='utf-8'))
rolling = json.loads(ROLL_PTR.read_text(encoding='utf-8'))
diagnosis = json.loads(DIAG_PTR.read_text(encoding='utf-8'))

assert dataset['status'] == 'PASS' and dataset['dataset_version'] == 'dataset_v012'
assert feature['status'] == 'PASS' and feature['pair_feature_version'] == 'pair_feature_v013'
assert selection['status'] == 'PASS' and selection['selection_version'] == 'feature_selection_v04'
assert rolling['status'] == 'PASS' and rolling['backtest_version'] == 'rolling_backtest_v01'
assert diagnosis['status'] == 'PASS' and diagnosis['diagnosis_version'] == 'underforecast_diagnosis_v01'
assert diagnosis['source_rolling_run_id'] == rolling['run_id']

PAIR_PANEL = Path(dataset['pair_panel_path'])
PAIR_FEATURE = Path(feature['pair_feature_panel_path'])
SELECTED = Path(selection['pair_selected_path'])
REF_PRED = Path(rolling['report_dir']) / 'rolling_origin_predictions.parquet'
for p in [PAIR_PANEL, PAIR_FEATURE, SELECTED, REF_PRED]:
    assert p.exists(), p

print('Accepted lineage:')
print(' Dataset   :', dataset['run_id'])
print(' Feature   :', feature['run_id'])
print(' Selection :', selection['run_id'])
print(' Rolling   :', rolling['run_id'])
print(' Diagnosis :', diagnosis['run_id'])
print(' Reference :', REF_PRED)

Accepted lineage:
 Dataset   : core_dataset_v012_20260815T122509Z
 Feature   : feature_stage_v013_20260815T123431Z
 Selection : feature_selection_v04_20260815T130048Z
 Rolling   : rolling_backtest_v01_20260815T132319Z
 Diagnosis : underforecast_diagnosis_v01_20260815T140720Z
 Reference : /content/drive/MyDrive/work9/06_reports/rolling_backtest/rolling_backtest_v01_20260815T132319Z/rolling_origin_predictions.parquet


## 2. Unit tests
These tests verify no hard threshold, exact same-row reference alignment, zero/positive diagnostics, and safety contract.

In [3]:
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', str(TEST)], text=True, capture_output=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise AssertionError(f'05C unit tests failed (returncode={r.returncode})')
print('05C unit tests PASS')

........                                                                 [100%]
8 passed in 2.69s

05C unit tests PASS


## 3. Run rolling challenger
Occurrence and positive-quantity components are fit separately for H1/H2/H3 using the same origin-honest FIT/CAL/EVAL policy as Stage 05. Evaluation labels are never used for fitting or early stopping.

In [4]:
import importlib.util
spec = importlib.util.spec_from_file_location('soft2', SRC)
soft2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(soft2)

run_id = 'soft_two_part_challenger_v01_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_dir = ROOT / '06_reports/soft_two_part_challenger' / run_id
manifest = soft2.run_soft_two_part_challenger(
    pair_panel_path=str(PAIR_PANEL),
    canonical_pair_feature_path=str(PAIR_FEATURE),
    selected_feature_path=str(SELECTED),
    model_contract_path=str(MODEL_CONTRACT),
    rolling_contract_path=str(ROLL_CONTRACT),
    challenger_contract_path=str(CHAL_CONTRACT),
    reference_prediction_path=str(REF_PRED),
    rolling_pointer=rolling,
    diagnosis_pointer=diagnosis,
    output_dir=str(out_dir),
    run_id=run_id,
    work9_root=str(ROOT),
)
print('05C PASS:', run_id)
print('Evaluation rows:', manifest['evaluation_rows'])
print('Selected features:', manifest['selected_feature_count'])
print('3M coverage:', manifest['cumulative_3m_coverage'])

[05C] origin=2025-01-01 fit=243,649 cal=53,475 eval=22,201
[05C] origin=2025-02-01 fit=288,597 cal=58,357 eval=23,690
[05C] origin=2025-03-01 fit=335,649 cal=62,921 eval=26,773
[05C] origin=2025-04-01 fit=383,851 cal=68,170 eval=30,075
[05C] origin=2025-05-01 fit=435,696 cal=73,603 eval=33,554
[05C] origin=2025-06-01 fit=488,894 cal=81,179 eval=35,700
[05C] origin=2025-07-01 fit=543,986 cal=90,058 eval=37,974
[05C] origin=2025-08-01 fit=601,914 cal=98,950 eval=39,699
[05C] origin=2025-09-01 fit=663,528 cal=106,560 eval=40,959
[05C] origin=2025-10-01 fit=728,652 cal=113,009 eval=42,632
[05C] origin=2025-11-01 fit=796,793 cal=118,438 eval=43,300
[05C] origin=2025-12-01 fit=867,380 cal=123,205 eval=44,289
05C PASS: soft_two_part_challenger_v01_20260815T145347Z
Evaluation rows: 420846
Selected features: 54
3M coverage: {'pair_origin_total': 142815, 'pair_origin_complete_h1_h2_h3': 135448, 'pair_origin_incomplete': 7367, 'complete_rate': 0.9484157826558834}


## 4. Same-row monthly + zero/positive comparison

In [5]:
score = pd.read_csv(out_dir / 'soft_two_part_scoreboard.csv')
zero_pos = pd.read_csv(out_dir / 'soft_two_part_zero_positive_diagnostics.csv')
prob = pd.read_csv(out_dir / 'soft_two_part_probability_diagnostics.csv')

print('Monthly overall and horizons:')
display(score[['model','horizon','wape','mae','bias_ratio','zero_false_positive_rate','positive_wape']])
print('Zero / positive decomposition:')
display(zero_pos[['model','horizon','actual_zero_rows','zero_forecast_sum_m2','actual_positive_rows','positive_wape','positive_bias_ratio','overall_wape','overall_bias_ratio']])
print('Occurrence / positive-quantity component diagnostics:')
display(prob)

Monthly overall and horizons:


,model,horizon,wape,mae,bias_ratio,zero_false_positive_rate,positive_wape
0,lightgbm_reference,NaN,1.009720,41.679839,-0.191949,1.0,0.729106
1,lightgbm_reference,1.0,0.968527,44.233778,-0.152887,1.0,0.710327
2,lightgbm_reference,2.0,1.022007,41.274391,-0.196367,1.0,0.732705
3,lightgbm_reference,3.0,1.046447,39.528864,-0.234496,1.0,0.747987
4,soft_two_part_expected,NaN,1.003091,41.406194,-0.176877,1.0,0.722192
5,soft_two_part_expected,1.0,0.956821,43.699153,-0.128675,1.0,0.700460
6,soft_two_part_expected,2.0,1.015902,41.027842,-0.184694,1.0,0.727798
7,soft_two_part_expected,3.0,1.045400,39.489335,-0.226854,1.0,0.742502


Zero / positive decomposition:


,model,horizon,actual_zero_rows,zero_forecast_sum_m2,actual_positive_rows,positive_wape,positive_bias_ratio,overall_wape,overall_bias_ratio
0,lightgbm_reference,NaN,264568,4.874809e+06,156278,0.729106,-0.472563,1.009720,-0.191949
1,soft_two_part_expected,NaN,264568,4.879759e+06,156278,0.722192,-0.457776,1.003091,-0.176877
2,lightgbm_reference,1.0,84294,1.655935e+06,56131,0.710327,-0.411087,0.968527,-0.152887
3,soft_two_part_expected,1.0,84294,1.644136e+06,56131,0.700460,-0.385035,0.956821,-0.128675
4,lightgbm_reference,2.0,88489,1.636831e+06,51607,0.732705,-0.485669,1.022007,-0.196367
5,soft_two_part_expected,2.0,88489,1.630054e+06,51607,0.727798,-0.472798,1.015902,-0.184694
6,lightgbm_reference,3.0,91785,1.582043e+06,48540,0.747987,-0.532956,1.046447,-0.234496
7,soft_two_part_expected,3.0,91785,1.605568e+06,48540,0.742502,-0.529752,1.045400,-0.226854


Occurrence / positive-quantity component diagnostics:


,horizon,n_rows,actual_positive_rate,mean_p_positive,mean_p_on_actual_zero,mean_p_on_actual_positive,probability_brier,positive_quantity_wape_on_positive_actual,positive_quantity_bias_ratio_on_positive_actual
0,NaN,420846,0.371342,0.353047,0.236280,0.550725,0.159178,0.707826,-0.146986
1,1.0,140425,0.399722,0.383639,0.238654,0.601370,0.154220,0.691262,-0.108515
2,2.0,140096,0.368369,0.347080,0.235923,0.537677,0.160972,0.715182,-0.150109
3,3.0,140325,0.345911,0.328390,0.234444,0.506033,0.162349,0.720015,-0.190199


## 5. 3M business aggregation + revision stability
These are computed from the **same forecast Pair universe** for both reference LightGBM and 05C challenger.

In [6]:
cum3 = pd.read_csv(out_dir / 'soft_two_part_cumulative_3m_scoreboard.csv')
rev = pd.read_csv(out_dir / 'soft_two_part_revision_scoreboard.csv')
stab = pd.read_csv(out_dir / 'soft_two_part_stability_summary.csv')

print('Cumulative 3M:')
display(cum3.sort_values(['level','wape_3m']))
print('Forecast revision:')
display(rev)
print('Rolling stability:')
display(stab)

review = json.loads((out_dir / 'soft_two_part_decision_review.json').read_text(encoding='utf-8'))
print(json.dumps(review, indent=2, ensure_ascii=False))

Cumulative 3M:


,model,level,n_units,wape_3m,mae_3m,bias_3m,bias_ratio_3m,actual_sum_m2,forecast_sum_m2
3,soft_two_part_expected,BASE_SKU,12837,0.440437,567.886810,-2.848379e+06,-0.172090,16551671.81,1.370329e+07
2,lightgbm_reference,BASE_SKU,12837,0.447575,577.091426,-3.120186e+06,-0.188512,16551671.81,1.343149e+07
5,soft_two_part_expected,BRANCH,655,0.291338,7362.035952,-2.848379e+06,-0.172090,16551671.81,1.370329e+07
4,lightgbm_reference,BRANCH,655,0.309998,7833.560619,-3.120186e+06,-0.188512,16551671.81,1.343149e+07
1,soft_two_part_expected,PAIR,135448,0.768839,93.951734,-2.848379e+06,-0.172090,16551671.81,1.370329e+07
0,lightgbm_reference,PAIR,135448,0.777706,95.035283,-3.120186e+06,-0.188512,16551671.81,1.343149e+07
7,soft_two_part_expected,PORTFOLIO,12,0.185014,255190.277160,-2.848379e+06,-0.172090,16551671.81,1.370329e+07
6,lightgbm_reference,PORTFOLIO,12,0.193325,266654.244037,-3.120186e+06,-0.188512,16551671.81,1.343149e+07


Forecast revision:


,model,level,transition,n_units,revision_mae_m2,revision_ratio_vs_old_forecast,signed_revision_m2
0,lightgbm_reference,PAIR,H2_TO_H1,125612,14.056912,0.412758,376769.438652
1,soft_two_part_expected,PAIR,H2_TO_H1,125612,14.803456,0.427257,424221.253486
2,lightgbm_reference,BASE_SKU,H2_TO_H1,11625,111.727631,0.303618,376769.438652
3,soft_two_part_expected,BASE_SKU,H2_TO_H1,11625,119.296355,0.318651,424221.253486
4,lightgbm_reference,PAIR,H3_TO_H2,125295,10.704456,0.361353,108060.009940
5,soft_two_part_expected,PAIR,H3_TO_H2,125295,10.841824,0.358863,57224.013504
6,lightgbm_reference,BASE_SKU,H3_TO_H2,11622,84.849699,0.265683,108060.009940
7,soft_two_part_expected,BASE_SKU,H3_TO_H2,11622,86.364896,0.265162,57224.013504


Rolling stability:


,model,n_origins,mean_origin_wape,median_origin_wape,std_origin_wape,mean_abs_bias_ratio,max_abs_bias_ratio,underforecast_origin_rate,origin_wins,pair_wape_3m,base_sku_wape_3m
0,soft_two_part_expected,12,1.004791,0.994815,0.054366,0.188474,0.329477,0.750000,7,0.768839,0.440437
1,lightgbm_reference,12,1.011860,1.000365,0.058278,0.195626,0.393351,0.916667,5,0.777706,0.447575


{
  "status": "REVIEW_REQUIRED",
  "auto_promote": false,
  "auto_freeze": false,
  "metrics": {
    "monthly_wape": {
      "lightgbm_reference": 1.0097204563712923,
      "soft_two_part_expected": 1.003091245508153
    },
    "monthly_bias_ratio": {
      "lightgbm_reference": -0.19194919605405938,
      "soft_two_part_expected": -0.1768771592211498
    },
    "cumulative_3m_pair": {
      "lightgbm_reference": {
        "wape_3m": 0.7777062747967908,
        "bias_ratio_3m": -0.18851185624540298
      },
      "soft_two_part_expected": {
        "wape_3m": 0.7688392220320148,
        "bias_ratio_3m": -0.17209010649437992
      }
    },
    "cumulative_3m_base_sku": {
      "lightgbm_reference": {
        "wape_3m": 0.44757549114750866,
        "bias_ratio_3m": -0.18851185624540295
      },
      "soft_two_part_expected": {
        "wape_3m": 0.4404366558769656,
        "bias_ratio_3m": -0.17209010649437995
      }
    },
    "cumulative_3m_branch": {
      "lightgbm_reference": {
  

## 6. Publish research pointer only
This pointer means **05C research PASS**, not champion promotion or model freeze.

In [7]:
assert manifest['status'] == 'PASS'
assert manifest['safety']['frozen_test_touched'] is False
assert manifest['safety']['model_freeze_run'] is False
assert manifest['safety']['production_published'] is False
assert manifest['safety']['hard_zero_threshold_used'] is False

ptr = {
    'run_id': run_id,
    'status': 'PASS',
    'challenger_version': 'soft_two_part_challenger_v01',
    'source_rolling_run_id': rolling['run_id'],
    'source_diagnosis_run_id': diagnosis['run_id'],
    'reference_model': 'lightgbm_tweedie',
    'challenger_model': 'soft_two_part_expected',
    'report_dir': str(out_dir),
    'manifest_path': str(out_dir / 'soft_two_part_manifest.json'),
    'decision_review_path': str(out_dir / 'soft_two_part_decision_review.json'),
}
ptr_path = ROOT / '01_config/current_soft_two_part_challenger_run.json'
ptr_path.write_text(json.dumps(ptr, ensure_ascii=False, indent=2), encoding='utf-8')
print('05C SOFT TWO-PART CHALLENGER PASS')
print('Pointer:', ptr_path)
print('STOP HERE. Send the 05C result for audit. Do NOT freeze or open Frozen Test yet.')

05C SOFT TWO-PART CHALLENGER PASS
Pointer: /content/drive/MyDrive/work9/01_config/current_soft_two_part_challenger_run.json
STOP HERE. Send the 05C result for audit. Do NOT freeze or open Frozen Test yet.
